In [1]:

import pandas as pd
from pathlib import Path

data_path = Path("../data/SMSSpamCollection")

df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

# Remove exact duplicate messages
df = df.drop_duplicates(
    subset="message"
).reset_index(drop=True)

# Convert labels into numbers
df["target"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["target"].value_counts())


Dataset shape: (5169, 3)

Class distribution:
target
0    4516
1     653
Name: count, dtype: int64


In [2]:

from sklearn.model_selection import train_test_split

X = df["message"]
y = df["target"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training messages:", len(X_train))
print("Validation messages:", len(X_val))


Training messages: 4135
Validation messages: 1034


In [3]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Logistic Regression model trained!")


Logistic Regression model trained!


In [4]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

y_pred = model.predict(X_val)
y_scores = model.predict_proba(X_val)[:, 1]

print("Classification Report:")
print(classification_report(
    y_val,
    y_pred,
    target_names=["Ham", "Spam"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("\nAverage Precision:")
print(average_precision_score(y_val, y_scores))


Classification Report:
              precision    recall  f1-score   support

         Ham       0.99      0.98      0.99       903
        Spam       0.90      0.92      0.91       131

    accuracy                           0.98      1034
   macro avg       0.94      0.95      0.95      1034
weighted avg       0.98      0.98      0.98      1034


Confusion Matrix:
[[889  14]
 [ 11 120]]

Average Precision:
0.9587706943479882


In [5]:

test_messages = [
    "Congratulations! You won a cash prize. Claim now!",
    "Hey, are you coming to college tomorrow?",
    "Your bank account has been suspended. Verify immediately."
]

predictions = model.predict(test_messages)
scores = model.predict_proba(test_messages)[:, 1]

for message, prediction, score in zip(
    test_messages, predictions, scores
):
    print("\nMessage:", message)
    print("Prediction:", "Spam" if prediction == 1 else "Ham")
    print("Spam probability estimate:", round(score, 3))



Message: Congratulations! You won a cash prize. Claim now!
Prediction: Spam
Spam probability estimate: 0.855

Message: Hey, are you coming to college tomorrow?
Prediction: Ham
Spam probability estimate: 0.125

Message: Your bank account has been suspended. Verify immediately.
Prediction: Spam
Spam probability estimate: 0.519


In [6]:

# Collect validation messages and predictions
errors = pd.DataFrame({
    "message": X_val.values,
    "actual": y_val.values,
    "predicted": y_pred,
    "spam_score": y_scores
})

# Keep only incorrect predictions
errors = errors[
    errors["actual"] != errors["predicted"]
]

print("Total misclassified:", len(errors))

print("\nFalse Negatives (Spam predicted as Ham):")
display(
    errors[
        (errors["actual"] == 1) &
        (errors["predicted"] == 0)
    ].head(10)
)

print("\nFalse Positives (Ham predicted as Spam):")
display(
    errors[
        (errors["actual"] == 0) &
        (errors["predicted"] == 1)
    ].head(10)
)


Total misclassified: 25

False Negatives (Spam predicted as Ham):


,message,actual,predicted,spam_score
95,Want explicit SEX in 30 secs? Ring 02073162414...,1,0,0.396759
230,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...,1,0,0.292714
308,Hi its LUCY Hubby at meetins all day Fri & I w...,1,0,0.213843
326,Missed call alert. These numbers called but le...,1,0,0.361671
437,Babe: U want me dont u baby! Im nasty and have...,1,0,0.227017
486,In The Simpsons Movie released in July 2007 na...,1,0,0.143582
690,"Latest News! Police station toilet stolen, cop...",1,0,0.327764
726,Want explicit SEX in 30 secs? Ring 02073162414...,1,0,0.351494
801,Would you like to see my XXX pics they are so ...,1,0,0.262562
826,ringtoneking 84484,1,0,0.227865



False Positives (Ham predicted as Spam):


,message,actual,predicted,spam_score
29,Your daily text from me – a favour this time,0,1,0.587145
155,I am waiting for your call sir.,0,1,0.667654
194,I'm vivek:)i got call from your number.,0,1,0.758451
293,Are you free now?can i call now?,0,1,0.702534
301,Please protect yourself from e-threats. SIB ne...,0,1,0.592374
305,"Sir, i am waiting for your call, once free ple...",0,1,0.823094
327,K k:) sms chat with me.,0,1,0.559397
533,Yes.. now only saw your message..,0,1,0.505295
737,Have you laid your airtel line to rest?,0,1,0.520860
749,Glad to see your reply.,0,1,0.608582


In [7]:

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2)
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

svm_pred = svm_model.predict(X_val)
svm_scores = svm_model.decision_function(X_val)

print("LinearSVC Results:")
print(classification_report(
    y_val,
    svm_pred,
    target_names=["Ham", "Spam"]
))

print("Average Precision:",
      average_precision_score(y_val, svm_scores))


LinearSVC Results:
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       903
        Spam       0.93      0.91      0.92       131

    accuracy                           0.98      1034
   macro avg       0.96      0.95      0.95      1034
weighted avg       0.98      0.98      0.98      1034

Average Precision: 0.9739095101294891


In [8]:

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "LinearSVC"],
    "Precision": [
        precision_score(y_val, y_pred),
        precision_score(y_val, svm_pred)
    ],
    "Recall": [
        recall_score(y_val, y_pred),
        recall_score(y_val, svm_pred)
    ],
    "F1": [
        f1_score(y_val, y_pred),
        f1_score(y_val, svm_pred)
    ],
    "Average Precision": [
        average_precision_score(y_val, y_scores),
        average_precision_score(y_val, svm_scores)
    ]
})

display(comparison.round(4))


,Model,Precision,Recall,F1,Average Precision
0,Logistic Regression,0.8955,0.9160,0.9057,0.9588
1,LinearSVC,0.9297,0.9084,0.9189,0.9739


In [9]:

from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    svm_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "ap": "average_precision"
    },
    n_jobs=-1
)

for metric in ["precision", "recall", "f1", "ap"]:
    values = scores[f"test_{metric}"]

    print(
        f"{metric}: "
        f"{values.mean():.4f} "
        f"(+/- {values.std():.4f})"
    )


precision: 0.9589 (+/- 0.0136)
recall: 0.9310 (+/- 0.0114)
f1: 0.9446 (+/- 0.0065)
ap: 0.9779 (+/- 0.0064)
